# Лабораторная работа № 3. Сбор данных через REST API

**Вариант 4:** DummyJSON Posts + Comments → DummyJSON Users.

## 1. Настройка

Импортируем библиотеки, задаём параметры запросов и создаём HTTP-сессию.
API публичный, аутентификация не требуется. Все результаты сохраняются рядом с блокнотом.

In [ ]:
import json
import random
import time
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
from pathlib import Path
from uuid import uuid4

import matplotlib.pyplot as plt
import pandas as pd
import requests
from IPython.display import display

BASE_URL = "https://dummyjson.com"
PAGE_SIZE, MAX_RECORDS, MAX_PAGES = 100, 1000, 20
TIMEOUT, PAUSE, MAX_ATTEMPTS = 30, 0.3, 4
OUT = Path('e:/Labs_2_course/Data_mining/lab_3')

session = requests.Session()
session.headers.update({"User-Agent": "University-Lab3/1.0", "Accept": "application/json"})
log_path = OUT / "collection_log.csv"
logs = pd.read_csv(log_path).to_dict("records") if log_path.exists() else []

## 2. Безопасные запросы и пагинация.

In [2]:
def parse_response(response):
    preview = response.content[:200].decode("utf-8", errors="replace")
    if not 200 <= response.status_code < 300:
        raise RuntimeError(f"HTTP {response.status_code}: {response.url}; {preview}")
    content_type = response.headers.get("Content-Type", "").split(";")[0].lower()
    if content_type != "application/json" and not content_type.endswith("+json"):
        raise ValueError(f"Ожидался JSON, получен {content_type}: {response.url}; {preview}")
    try:
        return response.json()
    except ValueError as error:
        raise ValueError(f"Некорректный JSON: {response.url}; {preview}") from error


def get_json(resource, params, page, run_id):
    url = f"{BASE_URL}/{resource}"
    for attempt in range(1, MAX_ATTEMPTS + 1):
        time.sleep(PAUSE)
        started = time.perf_counter()
        entry = {"run_id": run_id, "resource": resource, "page": page,
                 "attempt": attempt, "params": json.dumps(params), "url": url,
                 "timestamp": datetime.now(timezone.utc).isoformat(), "records": 0,
                 "status": None, "error": ""}
        try:
            response = session.get(url, params=params, timeout=TIMEOUT)
        except requests.RequestException as error:
            entry.update(duration_s=time.perf_counter() - started, error=str(error))
            logs.append(entry)
            pd.DataFrame(logs).to_csv(log_path, index=False)
            raise
        entry.update(status=response.status_code, url=response.url,
                     duration_s=time.perf_counter() - started,
                     retry_after=response.headers.get("Retry-After", ""),
                     rate_limit_headers=json.dumps({k: v for k, v in response.headers.items()
                                                    if "ratelimit" in k.lower()}))
        retryable = response.status_code == 429 or 500 <= response.status_code < 600
        if retryable and attempt < MAX_ATTEMPTS:
            entry["error"] = response.content[:200].decode("utf-8", errors="replace")
            logs.append(entry)
            pd.DataFrame(logs).to_csv(log_path, index=False)
            delay = 2 ** (attempt - 1) + random.uniform(0, 0.5)
            time.sleep(delay)
            continue
        try:
            data = parse_response(response)
            entry["records"] = len(data[resource])
        except (RuntimeError, ValueError, KeyError) as error:
            entry["error"] = str(error)
            raise
        finally:
            logs.append(entry)
            pd.DataFrame(logs).to_csv(log_path, index=False)
        return data


def paginate(resource, run_id, page_size=100, max_records=1000, max_pages=20, params=None):
    records, skip = [], 0
    for page in range(1, max_pages + 1):
        query = {**(params or {}), "limit": min(page_size, max_records - len(records)),
                 "skip": skip, "sortBy": "id", "order": "asc"}
        data = get_json(resource, query, page, run_id)
        with (OUT / "data_raw.jsonl").open("a", encoding="utf-8") as file:
            file.write(json.dumps({"run_id": run_id, "resource": resource, "page": page,
                                   "collected_at": datetime.now(timezone.utc).isoformat(),
                                   "response": data}, ensure_ascii=False) + "\n")
        batch = data[resource]
        records.extend(batch)
        skip += len(batch)
        if not batch or skip >= data["total"] or len(records) >= max_records:
            break
    return records

## 3. Нормализация и обогащение

In [3]:
def build_tables(comments, posts, users):
    comments_df = pd.json_normalize(comments, sep="_").rename(columns={
        "id": "comment_id", "body": "comment_text", "postId": "post_id",
        "likes": "comment_likes", "user_id": "commenter_id"})
    comments_df = comments_df[["comment_id", "comment_text", "post_id",
                               "comment_likes", "commenter_id"]]
    posts_df = pd.json_normalize(posts, sep="_").rename(columns={
        "id": "post_id", "title": "post_title", "userId": "post_author_id",
        "reactions_likes": "post_likes", "reactions_dislikes": "post_dislikes",
        "views": "post_views"})
    posts_df = posts_df[["post_id", "post_title", "post_author_id", "post_likes",
                         "post_dislikes", "post_views"]].drop_duplicates("post_id")
    users_df = pd.json_normalize(users, sep="_").rename(columns={
        "firstName": "first_name", "lastName": "last_name", "birthDate": "birth_date"})
    users_df = users_df.drop_duplicates("id")
    users_df["birth_date"] = pd.to_datetime(users_df["birth_date"], format="%Y-%m-%d",
                                            errors="coerce", utc=True)

    tags = pd.json_normalize(posts, record_path="tags", meta=["id", "userId"], sep="_")
    tags = tags.rename(columns={0: "tag", "id": "post_id", "userId": "post_author_id"})
    tags = tags.drop_duplicates(["post_id", "tag"])

    row_counts = {"исходные комментарии": len(comments_df)}
    clean = comments_df.merge(posts_df, on="post_id", how="left", validate="many_to_one")
    row_counts["после Posts"] = len(clean)
    for prefix, key in [("commenter", "commenter_id"), ("post_author", "post_author_id")]:
        lookup = users_df.add_prefix(prefix + "_")
        clean = clean.merge(lookup, on=key, how="left", validate="many_to_one")
        row_counts[f"после Users ({prefix})"] = len(clean)
    if len(set(row_counts.values())) != 1:
        raise ValueError(f"Изменилось число строк при обогащении: {row_counts}")
    for column in clean.columns:
        if column.endswith(("_id", "_likes", "_dislikes", "_views", "_age")):
            clean[column] = pd.to_numeric(clean[column], errors="coerce").astype("Int64")
    clean["comment_text"] = clean["comment_text"].str.strip()
    clean["post_title"] = clean["post_title"].str.strip()
    return clean, posts_df, users_df, tags, row_counts

## 4. Первый прогон и сохранение состояния

In [12]:
def run_pipeline():
    started = time.perf_counter()
    run_id = uuid4().hex[:12]
    state_path, clean_path = OUT / "state.json", OUT / "data_clean.csv"
    state = json.loads(state_path.read_text(encoding="utf-8")) if state_path.exists() else {}
    watermark = state.get("last_comment_id", 0)
    if watermark and not clean_path.exists():
        raise FileNotFoundError("state.json существует, но data_clean.csv отсутствует")

    options = dict(page_size=PAGE_SIZE, max_records=MAX_RECORDS, max_pages=MAX_PAGES)
    comments = paginate("comments", run_id, **options)
    posts = paginate("posts", run_id, **options)
    users = paginate("users", run_id, params={
        "select": "id,firstName,lastName,username,age,birthDate"}, **options)
    current, posts_df, users_df, tags, row_counts = build_tables(comments, posts, users)
    duplicate_count = int(current.duplicated("comment_id").sum())
    new = current[current["comment_id"] > watermark]
    old = pd.read_csv(clean_path) if clean_path.exists() else current.iloc[:0]
    new = new[~new["comment_id"].isin(old["comment_id"])]
    clean = pd.concat([old, new], ignore_index=True).drop_duplicates("comment_id")
    for column in clean.columns:
        if column.endswith(("_id", "_likes", "_dislikes", "_views", "_age")):
            clean[column] = pd.to_numeric(clean[column], errors="coerce").astype("Int64")
        elif column.endswith("_birth_date"):
            clean[column] = pd.to_datetime(clean[column], errors="coerce", utc=True)
    clean = clean.sort_values("comment_id").reset_index(drop=True)
    clean.to_csv(clean_path, index=False)
    tags.to_csv(OUT / "post_tags.csv", index=False)
    state_path.write_text(json.dumps({"last_comment_id": max(watermark, int(clean["comment_id"].max())),
                                      "updated_at": datetime.now(timezone.utc).isoformat()},
                                     ensure_ascii=False, indent=2), encoding="utf-8")
    run_log = pd.DataFrame([entry for entry in logs if entry["run_id"] == run_id])
    metrics = {"run_id": run_id, "records_fetched": len(comments), "new_records": len(new),
               "saved_records": len(clean), "posts": len(posts), "users": len(users),
               "comment_pages": int(run_log.loc[run_log["resource"] == "comments", "page"].nunique()),
               "requests": len(run_log), "duplicates_before": duplicate_count,
               "duration_s": round(time.perf_counter() - started, 3)}
    return clean, posts_df, users_df, metrics, row_counts


clean, posts_df, users_df, first_metrics, merge_counts = run_pipeline()
display(pd.Series(first_metrics, name="Первый прогон"))
display(clean.head())

run_id               a43ddf4019f3
records_fetched               340
new_records                     0
saved_records                 340
posts                         251
users                         208
comment_pages                   4
requests                       10
duplicates_before               0
duration_s                 17.892
Name: Первый прогон, dtype: object

,comment_id,comment_text,post_id,comment_likes,commenter_id,post_title,post_author_id,post_likes,post_dislikes,post_views,commenter_first_name,commenter_last_name,commenter_username,commenter_age,commenter_birth_date,post_author_first_name,post_author_last_name,post_author_username,post_author_age,post_author_birth_date
0,1,This is some awesome thinking!,242,3,105,The serene meadow stretched to the horizon,60,142,34,1197,Emma,Wilson,emmac,32,1993-06-02 00:00:00+00:00,Lillian,Simmons,lillians,33,1992-03-09 00:00:00+00:00
1,2,What terrific math skills you're showing!,46,4,183,"The clowns had taken over. And yes, they were ...",132,597,23,3730,Cameron,Perez,cameronp,34,1991-09-02 00:00:00+00:00,Zoe,Bennett,zoec,29,1996-05-11 00:00:00+00:00
2,3,You are an amazing writer!,235,2,1,The bustling city market was a feast for the s...,88,1457,15,3074,Emily,Johnson,emilys,29,1996-05-30 00:00:00+00:00,Aubrey,Wagner,aubreyw,31,1994-04-01 00:00:00+00:00
3,4,Wow! You have improved so much!,31,1,89,It was just a burger.,168,443,21,1656,Brayden,Fleming,braydenf,33,1992-07-20 00:00:00+00:00,Luna,Perez,lunah,27,1998-10-08 00:00:00+00:00
4,5,Nice idea!,212,1,149,The ancient ruins whispered tales of the past,130,603,38,3061,Wyatt,Perry,wyattp,35,1990-05-23 00:00:00+00:00,Chloe,Roberts,chloep,42,1983-04-08 00:00:00+00:00


## 5. Качество данных

Для каждого поля показываем полноту и тип, считаем дубликаты. Проверяем
неотрицательность счётчиков, существование постов и авторов, непустой текст
и даты рождения не позже текущей даты. Числовые выбросы ищем методом IQR;
такие наблюдения оставляем, поскольку высокая активность может быть допустимой.

In [14]:
quality = pd.DataFrame({"field": clean.columns,
                        "completeness_pct": (clean.notna().mean() * 100).round(2).values,
                        "dtype": clean.dtypes.astype(str).values})
quality.to_csv(OUT / "quality.csv", index=False)
display(quality)
print("Дубликаты до удаления:", first_metrics["duplicates_before"])
print("Дубликаты в итоговой таблице:", clean.duplicated("comment_id").sum())

counts = ["comment_likes", "post_likes", "post_dislikes", "post_views"]
now = pd.Timestamp.now(tz="UTC")
rules = {
    "Комментарий связан с существующим постом": clean["post_id"].isin(posts_df["post_id"]),
    "Авторы существуют в Users": clean["commenter_id"].isin(users_df["id"])
        & clean["post_author_id"].isin(users_df["id"]),
    "Даты рождения не в будущем": clean["commenter_birth_date"].le(now)
        & clean["post_author_birth_date"].le(now),
}
checks = pd.DataFrame([{"check": name, "violations": int((~valid).sum())}
                       for name, valid in rules.items()])
checks.to_csv(OUT / "business_checks.csv", index=False)
display(checks)

,field,completeness_pct,dtype
0,comment_id,100.0,Int64
1,comment_text,100.0,object
2,post_id,100.0,Int64
3,comment_likes,100.0,Int64
4,commenter_id,100.0,Int64
5,post_title,100.0,object
6,post_author_id,100.0,Int64
7,post_likes,100.0,Int64
8,post_dislikes,100.0,Int64
9,post_views,100.0,Int64


Дубликаты до удаления: 0
Дубликаты в итоговой таблице: 0


,check,violations
0,Комментарий связан с существующим постом,0
1,Авторы существуют в Users,0
2,Даты рождения не в будущем,0


## 6. Активность авторов и связь комментариев с реакциями

Считаем комментарии и их лайки на пост, включая посты без комментариев.
Реакции постов учитываем один раз на пост, а не суммируем повторённые значения
из основной таблицы. Для авторов считаем число постов, написанных и полученных
комментариев. Строим графики и вычисляем корреляцию числа комментариев с лайками поста.

In [ ]:
comment_stats = clean.groupby("post_id").agg(
    comments_count=("comment_id", "count"), comment_likes_sum=("comment_likes", "sum"))
posts_activity = posts_df.merge(comment_stats, on="post_id", how="left", validate="one_to_one")
posts_activity[["comments_count", "comment_likes_sum"]] = posts_activity[
    ["comments_count", "comment_likes_sum"]].fillna(0).astype(int)
posts_activity.to_csv(OUT / "posts_activity.csv", index=False)

written = clean.groupby("commenter_id").size().rename("written_comments")
authored = posts_activity.groupby("post_author_id").agg(
    posts_count=("post_id", "count"), received_comments=("comments_count", "sum"),
    post_likes_sum=("post_likes", "sum"))
authors_activity = users_df[["id", "first_name", "last_name", "username"]].merge(
    written, left_on="id", right_index=True, how="left", validate="one_to_one").merge(
    authored, left_on="id", right_index=True, how="left", validate="one_to_one")
activity_fields = ["written_comments", "posts_count", "received_comments", "post_likes_sum"]
authors_activity[activity_fields] = authors_activity[activity_fields].fillna(0).astype(int)
authors_activity = authors_activity.sort_values("written_comments", ascending=False)
authors_activity.to_csv(OUT / "authors_activity.csv", index=False)
display(authors_activity.head(10))

,id,first_name,last_name,username,written_comments,posts_count,received_comments,post_likes_sum
82,83,Dylan,Wells,dylanw,7,3,6,2181
38,39,Lucas,Gray,lucasg,6,0,0,0
24,25,Harper,Kelly,harpere,5,0,0,0
13,14,Charlotte,Lopez,charlottem,5,0,0,0
73,74,Penelope,Harper,penelopeh,5,1,0,1
50,51,Eli,Bennett,elib,5,3,9,2483
56,57,Nathan,Dixon,nathand,4,2,2,986
170,171,Connor,Baker,connorb,4,2,4,1760
145,146,Leah,Hughes,leahp,4,0,0,0
115,116,William,Lopez,williaml,4,1,3,1028


## 7. Инкрементальный повторный запуск

In [7]:
second_clean, _, _, second_metrics, _ = run_pipeline()
metrics = pd.DataFrame([first_metrics, second_metrics])
metrics.to_csv(OUT / "run_metrics.csv", index=False)
display(metrics)
print("Изменение числа строк:", len(second_clean) - len(clean))
print("Новые записи во втором прогоне:", second_metrics["new_records"])
print("Дубликаты после повторного запуска:", second_clean.duplicated("comment_id").sum())
display(pd.DataFrame(logs).tail())
session.close()

,run_id,records_fetched,new_records,saved_records,posts,users,comment_pages,requests,duplicates_before,duration_s
0,52e7e7c8180d,340,0,340,251,208,4,10,0,17.399
1,db89e03adc21,340,0,340,251,208,4,10,0,5.975


Изменение числа строк: 0
Новые записи во втором прогоне: 0
Дубликаты после повторного запуска: 0


,run_id,resource,page,attempt,params,url,timestamp,records,status,error,duration_s,retry_after,rate_limit_headers
35,db89e03adc21,posts,2,1,"{""limit"": 100, ""skip"": 100, ""sortBy"": ""id"", ""o...",https://dummyjson.com/posts?limit=100&skip=100...,2026-10-06T13:39:46.685867+00:00,100,200,,0.384943,,"{""x-ratelimit-limit"": ""100"", ""x-ratelimit-rema..."
36,db89e03adc21,posts,3,1,"{""limit"": 100, ""skip"": 200, ""sortBy"": ""id"", ""o...",https://dummyjson.com/posts?limit=100&skip=200...,2026-10-06T13:39:47.393050+00:00,51,200,,0.248758,,"{""x-ratelimit-limit"": ""100"", ""x-ratelimit-rema..."
37,db89e03adc21,users,1,1,"{""select"": ""id,firstName,lastName,username,age...",https://dummyjson.com/users?select=id%2CfirstN...,2026-10-06T13:39:47.947862+00:00,100,200,,0.246318,,"{""x-ratelimit-limit"": ""100"", ""x-ratelimit-rema..."
38,db89e03adc21,users,2,1,"{""select"": ""id,firstName,lastName,username,age...",https://dummyjson.com/users?select=id%2CfirstN...,2026-10-06T13:39:48.502900+00:00,100,200,,0.256180,,"{""x-ratelimit-limit"": ""100"", ""x-ratelimit-rema..."
39,db89e03adc21,users,3,1,"{""select"": ""id,firstName,lastName,username,age...",https://dummyjson.com/users?select=id%2CfirstN...,2026-10-06T13:39:49.085805+00:00,8,200,,0.242773,,"{""x-ratelimit-limit"": ""100"", ""x-ratelimit-rema..."
